# 2 · Interactive dashboard **inside your notebook** | 25 minutes
Build visuals directly in Colab/Jupyter with Plotly. No Streamlit server yet. You are querying **the real data you cleaned in Notebook 1**. The instructor or your team should have loaded the team table first.

**Analyst mindset:** Write one finding and one caveat based on your charts.

In [ ]:
%pip install -q "psycopg[binary]>=3.2,<4" "pandas>=2.2,<3" "plotly>=5,<7" "openai>=1.55,<3"

In [ ]:
import os, getpass
from contextlib import closing
import psycopg
from psycopg import sql
import pandas as pd

# Reenter these in EACH notebook. Use the team credential supplied privately by your instructor.
PROJECT_REF = os.getenv("LAB_PROJECT_REF") or input("Supabase project ref: ").strip()
POOLER_HOST = os.getenv("LAB_POOLER_HOST") or input("Transaction pooler host (not full URL): ").strip()
TEAM_NUMBER = int(os.getenv("LAB_TEAM_NUMBER") or input("Team number (e.g. 1): ").strip())
TEAM_SCHEMA = f"team_{TEAM_NUMBER:02d}"
DB_PASSWORD = os.getenv("LAB_DB_PASSWORD") or getpass.getpass("Team database password: ")
DB_USER = f"lab_team_{TEAM_NUMBER:02d}.{PROJECT_REF}"

def connect():
    # Supabase Free transaction pooler; psycopg prepared statements disabled.
    return psycopg.connect(host=POOLER_HOST, port=6543, dbname="postgres",
                           user=DB_USER, password=DB_PASSWORD, sslmode="require",
                           connect_timeout=12, prepare_threshold=None)

def fetch_df(query, params=None):
    with closing(connect()) as conn:
        with conn.cursor() as cur:
            cur.execute(query, params)
            if cur.description is None: return pd.DataFrame()
            return pd.DataFrame(cur.fetchall(), columns=[x.name for x in cur.description])

def run_sql(query, params=None):
    with closing(connect()) as conn:
        with conn.transaction():
            with conn.cursor() as cur:
                cur.execute(query, params)

print("Connected to:", TEAM_SCHEMA)
print(fetch_df("SELECT current_user AS user_name, current_database() AS db_name"))

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display
import plotly.io as pio
pio.renderers.default = "notebook_connected"


## Load your clean tables and understand the data

In [ ]:
data = fetch_df(f"SELECT * FROM {TEAM_SCHEMA}.clean_tickets")
assert len(data)>0, "Run Notebook 1 ETL first, or ask instructor for recovery."
display(data.head(5)); print("Rows:",len(data))

In [ ]:
def metrics_by_type():
    return fetch_df(f"""
        SELECT ticket_type, COUNT(*) AS tickets,
               ROUND(AVG(response_to_resolution_hours),2) AS avg_response_to_resolution_hours,
               ROUND(AVG(satisfaction_rating),2) AS avg_satisfaction
        FROM {TEAM_SCHEMA}.clean_tickets
        GROUP BY ticket_type ORDER BY tickets DESC
    """)

def summary_metrics():
    return fetch_df(f"""
        SELECT COUNT(*) AS total_tickets,
               COUNT(*) FILTER (WHERE status = 'Closed') AS closed_tickets,
               ROUND(AVG(satisfaction_rating),2) AS avg_satisfaction,
               ROUND(AVG(response_to_resolution_hours),2) AS avg_response_to_resolution_hours
        FROM {TEAM_SCHEMA}.clean_tickets
    """)

## TODO 1 · KPIs: show what matters

In [ ]:
stats = summary_metrics().iloc[0]
closed_rate = (100 * stats["closed_tickets"] / stats["total_tickets"]) if stats["total_tickets"] else 0
print(f"Total tickets: {stats['total_tickets']:,}")
print(f"Closed rate: {closed_rate:.1f}%")
print(f"Average satisfaction: {stats['avg_satisfaction']}")

# Dynamic KPI Scorecard
from plotly.subplots import make_subplots

fig_kpi = make_subplots(
    rows=1, cols=4,
    specs=[[{"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}]]
)

fig_kpi.add_trace(go.Indicator(
    mode="number",
    value=int(stats["total_tickets"]),
    title={"text": "<b>Total Tickets</b>", "font": {"size": 15}},
    number={"font": {"color": "#1E3A8A"}}
), row=1, col=1)

fig_kpi.add_trace(go.Indicator(
    mode="number+gauge",
    value=float(closed_rate),
    number={"suffix": "%", "font": {"color": "#047857"}},
    title={"text": "<b>Resolution Rate</b>", "font": {"size": 15}},
    gauge={"axis": {"range": [0, 100]}, "bar": {"color": "#10B981"}}
), row=1, col=2)

fig_kpi.add_trace(go.Indicator(
    mode="number+gauge",
    value=float(stats["avg_satisfaction"] or 0),
    number={"valueformat": ".2f", "font": {"color": "#D97706"}},
    title={"text": "<b>Avg CSAT (1-5)</b>", "font": {"size": 15}},
    gauge={"axis": {"range": [1, 5]}, "bar": {"color": "#F59E0B"}}
), row=1, col=3)

fig_kpi.add_trace(go.Indicator(
    mode="number",
    value=float(stats["avg_response_to_resolution_hours"] or 0),
    number={"suffix": "h", "valueformat": ".1f", "font": {"color": "#6366F1"}},
    title={"text": "<b>Avg Resolution</b>", "font": {"size": 15}}
), row=1, col=4)

fig_kpi.update_layout(
    height=160,
    margin=dict(l=20, r=20, t=25, b=20),
    paper_bgcolor="#F8FAFC"
)
fig_kpi.show()


## TODO 2 · Build two different charts

In [ ]:
summary = metrics_by_type()

# 1. QUADRANT BUBBLE MATRIX: Resolution Speed vs Satisfaction
# Reveals: Which ticket categories take the longest and hurt CSAT the most
fig_quad = px.scatter(
    summary,
    x="avg_response_to_resolution_hours",
    y="avg_satisfaction",
    size="tickets",
    color="ticket_type",
    text="ticket_type",
    title="<b>1. Support Quadrant Matrix: Resolution Hours vs Customer Satisfaction</b><br><sup>Bubble Size = Ticket Count | Top-Left = Fast & High CSAT | Bottom-Right = Critical Bottlenecks</sup>",
    labels={
        "avg_response_to_resolution_hours": "Average Resolution Time (Hours)",
        "avg_satisfaction": "Average Customer Satisfaction (1-5)",
        "tickets": "Ticket Count"
    },
    template="plotly_white"
)
fig_quad.update_traces(textposition="top center", marker=dict(opacity=0.85, line=dict(width=1.5, color="DarkSlateGrey")))
# Benchmark lines
med_hours = summary["avg_response_to_resolution_hours"].median()
fig_quad.add_vline(x=med_hours, line_dash="dash", line_color="gray", annotation_text=f"Median Time ({med_hours:.1f}h)")
fig_quad.add_hline(y=3.0, line_dash="dash", line_color="red", annotation_text="CSAT Baseline (3.0)")
fig_quad.update_layout(height=480)
fig_quad.show()

# 2. HIERARCHICAL SUNBURST: Interactive Multi-Level Drilldown
# Reveals: Click any slice to zoom into Channel -> Priority -> Status
fig_sun = px.sunburst(
    data,
    path=["channel", "priority", "status"],
    title="<b>2. Interactive Multi-Level Drilldown: Channel → Priority → Status</b><br><sup>Click on any ring segment to zoom in / click center to zoom out</sup>",
    color="priority",
    color_discrete_map={"Critical": "#EF4444", "High": "#F97316", "Medium": "#EAB308", "Low": "#10B981"},
    template="plotly_white"
)
fig_sun.update_layout(height=520, margin=dict(t=50, l=10, r=10, b=10))
fig_sun.show()

# 3. DISTRIBUTION & OUTLIER BOX PLOT: Resolution Time Spread by Priority
# Reveals: Averages hide extreme delays; box plot shows real median and outliers
clean_durations = data.dropna(subset=["response_to_resolution_hours"])
fig_box = px.box(
    clean_durations,
    x="priority",
    y="response_to_resolution_hours",
    color="priority",
    points="outliers",
    category_orders={"priority": ["Low", "Medium", "High", "Critical"]},
    title="<b>3. Resolution Time Spread & Outliers by Ticket Priority</b><br><sup>Inspects actual variance and long-tail resolution bottlenecks</sup>",
    labels={"response_to_resolution_hours": "Resolution Time (Hours)", "priority": "Priority"},
    template="plotly_white"
)
fig_box.update_layout(height=450, showlegend=False)
fig_box.show()


## Explore a filter and chart quality

In [ ]:
# 4. CROSS-CHANNEL HEATMAP: Issue Type vs Support Channel
channel_cross = pd.crosstab(data["ticket_type"], data["channel"])
fig_heat = px.imshow(
    channel_cross,
    text_auto=True,
    aspect="auto",
    color_continuous_scale="Blues",
    title="<b>4. Issue-to-Channel Volume Heatmap</b><br><sup>Where do specific customer issues land? (Darker = Higher concentration)</sup>",
    labels=dict(x="Support Channel", y="Ticket Type", color="Tickets"),
    template="plotly_white"
)
fig_heat.update_layout(height=400)
fig_heat.show()

# 5. DYNAMIC INTERACTIVE CHANNEL EXPLORER (Dropdown Metric Switcher)
# Switch between Volume, Satisfaction, and Resolution Time without re-running code!
channel_metrics = fetch_df(f"""
    SELECT channel, 
           COUNT(*) AS tickets,
           ROUND(AVG(satisfaction_rating), 2) AS avg_satisfaction,
           ROUND(AVG(response_to_resolution_hours), 2) AS avg_resolution_hours
    FROM {TEAM_SCHEMA}.clean_tickets
    GROUP BY channel
    ORDER BY tickets DESC
""")

fig_dyn = go.Figure()
fig_dyn.add_trace(go.Bar(
    x=channel_metrics["channel"],
    y=channel_metrics["tickets"],
    name="Ticket Volume",
    text=channel_metrics["tickets"],
    textposition="auto",
    marker_color="#2563EB",
    visible=True
))
fig_dyn.add_trace(go.Bar(
    x=channel_metrics["channel"],
    y=channel_metrics["avg_satisfaction"],
    name="Avg Satisfaction (1-5)",
    text=channel_metrics["avg_satisfaction"],
    textposition="auto",
    marker_color="#059669",
    visible=False
))
fig_dyn.add_trace(go.Bar(
    x=channel_metrics["channel"],
    y=channel_metrics["avg_resolution_hours"],
    name="Avg Resolution Hours",
    text=channel_metrics["avg_resolution_hours"],
    textposition="auto",
    marker_color="#D97706",
    visible=False
))

fig_dyn.update_layout(
    title="<b>5. Dynamic Channel Explorer</b> (Use dropdown below to switch view)",
    template="plotly_white",
    height=450,
    updatemenus=[
        dict(
            active=0,
            buttons=[
                dict(
                    label="📊 Ticket Volume",
                    method="update",
                    args=[{"visible": [True, False, False]},
                          {"title": "<b>Ticket Volume by Channel</b>", "yaxis": {"title": "Number of Tickets"}}]
                ),
                dict(
                    label="⭐ Customer Satisfaction (CSAT)",
                    method="update",
                    args=[{"visible": [False, True, False]},
                          {"title": "<b>Average Customer Satisfaction by Channel (1-5)</b>", "yaxis": {"title": "CSAT Rating"}}]
                ),
                dict(
                    label="⏱️ Avg Resolution Hours",
                    method="update",
                    args=[{"visible": [False, False, True]},
                          {"title": "<b>Average Resolution Hours by Channel</b>", "yaxis": {"title": "Hours"}}]
                ),
            ],
            direction="down",
            pad={"r": 10, "t": 10},
            showactive=True,
            x=0.0,
            xanchor="left",
            y=1.22,
            yanchor="top"
        )
    ],
    xaxis_title="Support Channel",
    yaxis_title="Number of Tickets"
)
fig_dyn.show()


### Checkpoint
Write down: Which ticket type has the highest volume? Does average satisfaction differ by type? Why are null response-to-resolution values excluded from AVG? Do not label the timestamp-derived metric as contractual SLA compliance.